In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")

In [0]:
# Clears all cached Spark DataFrames in the session
spark.catalog.clearCache()

In [0]:
Total_Population = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("TicketFLAG")== "Yes")
    .filter(F.col("CineClubFLAG") == "No")
    .withColumn("Year", F.year(F.col("datefull")))
    .groupBy("Year")
    .agg(
        F.sum("Quantity").alias("Total_Tickets"),
        F.sum("Net_Revenue").alias("Net_Revenue_Tickets")
    )
)

In [0]:
Total_Population.display()

In [0]:
Total_Population_Conc = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .join(T_locations.select("LocationID", "LC_Location_Type_Description"), T_trans.LocationId == T_locations.LocationID, "left")
    .filter(F.col("datefull").between("2019-01-01", "2025-12-31"))
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("ConcessionFLAG")== "Yes")
    .filter(F.col("CineClubFLAG") == "No")
    .withColumn("Year", F.year(F.col("datefull")))
    .groupBy("Year")
    .agg(
        F.sum("Net_Revenue").alias("Net_Revenue_Concessions")
    )
)

In [0]:
    Total_Population_Conc.display()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

years = [2021, 2022, 2023, 2024, 2025]  # edit as needed

outs = []

for Y in years:
    start_id = Y*10000 + 101       # YYYY0101
    end_id   = (Y+1)*10000 + 101   # (YYYY+1)0101 (exclusive)

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_id) & (F.col("VisitDateID") < end_id))
        .select("cde_id", "CineClubFLAG")  # keep it light
    )

    non_cineclub_ids_y = (
        base_y
        .filter(F.col("CineClubFLAG") != "Yes")   # matches your table (NULL counts as non-CC)
        .select("cde_id")
        .distinct()
    )

    cc_only_ids_y = (
        base_y
        .filter(F.col("CineClubFLAG") == "Yes")
        .select("cde_id")
        .distinct()
        .join(non_cineclub_ids_y, "cde_id", "left_anti")
    )

    out_y = (
        cc_only_ids_y
        .agg(F.countDistinct("cde_id").alias("Members"))
        .withColumn("Year", F.lit(Y))
        .select("Year", "Members")
    )

    outs.append(out_y)

df_members_by_year = reduce(lambda a, b: a.unionByName(b), outs)
display(df_members_by_year.orderBy("Year"))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

years = [2021,2022,2023,2024,2025]  # edit as needed
outs = []

for Y in years:
    start_id = Y*10000 + 101       # YYYY0101
    end_id   = (Y+1)*10000 + 101   # (YYYY+1)0101 (exclusive)

    base_y = (
        T_trans
        .filter((F.col("VisitDateID") >= start_id) & (F.col("VisitDateID") < end_id))
        .select(
            "cde_id", "CineClubFLAG", "VisitDateID", "LocationId",
            "TicketFLAG", "ConcessionFLAG", "Net_Revenue"
        )
    )

    # Cohort: CC-only within the year (NULL treated as non-CC, matching your table)
    non_cineclub_ids_y = (
        base_y
        .filter(F.col("CineClubFLAG") != "Yes")
        .select("cde_id")
        .distinct()
    )

    cc_only_ids_y = (
        base_y
        .filter(F.col("CineClubFLAG") == "Yes")
        .select("cde_id")
        .distinct()
        .join(non_cineclub_ids_y, "cde_id", "left_anti")
    )

    # Transactions for those members in that year
    tx_y = base_y.join(cc_only_ids_y, "cde_id", "inner")

    # Visits = distinct (cde_id, VisitDateID, LocationId)
    visits_y = tx_y.select("cde_id", "VisitDateID", "LocationId").distinct()

    out_y = (
        tx_y
        .agg(
            F.countDistinct("cde_id").alias("Members"),

            # Visits (distinct by your definition)
            F.countDistinct(F.struct("cde_id", "VisitDateID", "LocationId")).alias("Total_Visits"),

            # Net revenues
            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Net_Ticket_Revenue"),
            F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Net_Concession_Revenue"),
        )
        .withColumn("Year", F.lit(Y))
        .select("Year", "Members", "Total_Visits", "Net_Ticket_Revenue", "Net_Concession_Revenue")
    )

    outs.append(out_y)

df_yearly = reduce(lambda a, b: a.unionByName(b), outs)
display(df_yearly.orderBy("Year"))

In [0]:
from pyspark.sql import functions as F

# Years to compare
Y_main = 2025
Y_prev = 2023

start_main = Y_main*10000 + 101
end_main   = (Y_main+1)*10000 + 101

start_prev = Y_prev*10000 + 101
end_prev   = (Y_prev+1)*10000 + 101

# ----------------------------
# Cohort builder (same definition you’ve been using)
# CC-only within the year; NULL CineClubFLAG treated as non-CC because of != "Yes"
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_main) & (F.col("VisitDateID") < end_main))
    .select("cde_id", "CineClubFLAG")
)

non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

base_2023 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_prev) & (F.col("VisitDateID") < end_prev))
    .select("cde_id", "CineClubFLAG")
)

non_cc_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") != "Yes")
    .select("cde_id").distinct()
)

cc_only_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2023, "cde_id", "left_anti")
)

# ----------------------------
# Overlap: 2025 members who are also members in 2023
# ----------------------------
overlap_25_23 = cc_only_2025.join(cc_only_2023, "cde_id", "inner")

# Counts
counts = (
    cc_only_2025.agg(F.countDistinct("cde_id").alias("Members_2025"))
    .crossJoin(cc_only_2023.agg(F.countDistinct("cde_id").alias("Members_2023")))
    .crossJoin(overlap_25_23.agg(F.countDistinct("cde_id").alias("Members_2025_also_2023")))
)

# If you also want the actual list of overlapping members:
# display(overlap_25_23)

In [0]:
display(counts)

In [0]:
from pyspark.sql import functions as F

# ----------------------------
# Year params
# ----------------------------
Y_main = 2025
Y_prev = 2023

start_main = Y_main*10000 + 101
end_main   = (Y_main+1)*10000 + 101

start_prev = Y_prev*10000 + 101
end_prev   = (Y_prev+1)*10000 + 101

# ----------------------------
# 1) 2025 cohort = CC-only within 2025 (your same definition)
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_main) & (F.col("VisitDateID") < end_main))
    .select("cde_id", "CineClubFLAG")
)

non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")   # NULL treated as non-CC (matches your table)
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

# ----------------------------
# 2) 2023 presence + CC participation flags
# ----------------------------
base_2023 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_prev) & (F.col("VisitDateID") < end_prev))
    .select("cde_id", "CineClubFLAG")
)

# Everyone who "exists" in 2023 (any transaction at all)
exists_2023 = base_2023.select("cde_id").distinct()

# Anyone who was a CC member at least once in 2023 (at least one Yes)
cc_any_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
)

# Exists in 2023 but NOT CC at all in 2023
exists_2023_not_cc = exists_2023.join(cc_any_2023, "cde_id", "left_anti")

# ----------------------------
# 3) Required counts for the 2025 cohort
# ----------------------------
# (1) 2025 cohort members who exist in 2023 but were NOT CC at all in 2023
cnt_exist_2023_not_cc = (
    cc_only_2025
    .join(exists_2023_not_cc, "cde_id", "inner")
    .agg(F.countDistinct("cde_id").alias("Members_2025_exist_2023_not_CC_2023"))
)

# (2) 2025 cohort members who did NOT exist in 2023 at all
cnt_not_exist_2023 = (
    cc_only_2025
    .join(exists_2023, "cde_id", "left_anti")
    .agg(F.countDistinct("cde_id").alias("Members_2025_not_exist_in_2023"))
)

# Optional: also show the 2025 cohort size for context
cnt_2025 = cc_only_2025.agg(F.countDistinct("cde_id").alias("Members_2025_CC_only"))

# Final output table
final = (
    cnt_2025
    .crossJoin(cnt_exist_2023_not_cc)
    .crossJoin(cnt_not_exist_2023)
)

display(final)

In [0]:
from pyspark.sql import functions as F

# ----------------------------
# Years
# ----------------------------
Y_main = 2025
Y_prev = 2021

start_2025 = Y_main*10000 + 101
end_2025   = (Y_main+1)*10000 + 101

start_2023 = Y_prev*10000 + 101
end_2023   = (Y_prev+1)*10000 + 101

# ----------------------------
# 0) Base pulls (only needed cols)
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
)

base_2023 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2023) & (F.col("VisitDateID") < end_2023))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
)

# ----------------------------
# 1) Build the cohort: Members_2025_exist_2023_not_CC_2023
#    (same logic as before)
# ----------------------------
non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")     # NULL treated as non-CC
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

exists_2023 = base_2023.select("cde_id").distinct()

cc_any_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
)

exists_2023_not_cc = exists_2023.join(cc_any_2023, "cde_id", "left_anti")

Members_2025_exist_2023_not_CC_2023 = (
    cc_only_2025.join(exists_2023_not_cc, "cde_id", "inner")
)

# ----------------------------
# 2) Metrics for that cohort in 2023 and 2025
#    Visit definition: distinct (cde_id, VisitDateID, LocationId)
# ----------------------------
tx_2023 = base_2023.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")
tx_2025 = base_2025.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")

m_2023 = tx_2023.agg(
    F.countDistinct("cde_id").alias("Members"),
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2023"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2023"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2023"),
)

m_2025 = tx_2025.agg(
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2025"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2025"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2025"),
)

final = (
    m_2023.crossJoin(m_2025)
    .select(
        "Members",
        "Visits_2023","Visits_2025",
        "Ticket_Revenue_2023","Ticket_Revenue_2025",
        "Concession_Revenue_2023","Concession_Revenue_2025"
    )
)

display(final)

In [0]:
from pyspark.sql import functions as F

# ----------------------------
# Years
# ----------------------------
Y_main = 2025
Y_prev = 2019

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")

start_2025 = Y_main*10000 + 101
end_2025   = (Y_main+1)*10000 + 101

start_2023 = Y_prev*10000 + 101
end_2023   = (Y_prev+1)*10000 + 101

# ----------------------------
# 0) Base pulls (only needed cols)
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue", "Quantity")
)

base_2023 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2023) & (F.col("VisitDateID") < end_2023))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue", "Quantity")
)

# ----------------------------
# 1) Build the cohort: Members_2025_exist_2023_not_CC_2023
#    (same logic as before)
# ----------------------------
non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")     # NULL treated as non-CC
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

exists_2023 = base_2023.select("cde_id").distinct()

cc_any_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
)

exists_2023_not_cc = exists_2023.join(cc_any_2023, "cde_id", "left_anti")

Members_2025_exist_2023_not_CC_2023 = (
    cc_only_2025.join(exists_2023_not_cc, "cde_id", "inner")
)

# ----------------------------
# 2) Metrics for that cohort in 2023 and 2025
#    Visit definition: distinct (cde_id, VisitDateID, LocationId)
# ----------------------------
tx_2023 = base_2023.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")
tx_2025 = base_2025.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")

m_2023 = tx_2023.agg(
    F.countDistinct("cde_id").alias("Members"),
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2023"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2023"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Quantity"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Count_2023"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2023"),
)

m_2025 = tx_2025.agg(
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2025"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2025"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Quantity"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Count_2025"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2025"),
)

final = (
    m_2023.crossJoin(m_2025)
    .select(
        "Members",
        "Visits_2023","Visits_2025",
        "Ticket_Revenue_2023","Ticket_Revenue_2025",
        "Concession_Revenue_2023","Concession_Revenue_2025",
        "Ticket_Count_2023", "Ticket_Count_2025"
    )
)

display(final)

In [0]:
from pyspark.sql import functions as F

# ----------------------------
# Years
# ----------------------------
Y_main = 2025
Y_prev = 2024

start_2025 = Y_main*10000 + 101
end_2025   = (Y_main+1)*10000 + 101

start_2023 = Y_prev*10000 + 101
end_2023   = (Y_prev+1)*10000 + 101

# ----------------------------
# 0) Base pulls (only needed cols)
# ----------------------------
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
)

base_2023 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2023) & (F.col("VisitDateID") < end_2023))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
)

# ----------------------------
# 1) Build the cohort: Members_2025_exist_2023_not_CC_2023
#    (same logic as before)
# ----------------------------
non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")     # NULL treated as non-CC
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

exists_2023 = base_2023.select("cde_id").distinct()

cc_any_2023 = (
    base_2023
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
)

exists_2023_not_cc = exists_2023.join(cc_any_2023, "cde_id", "left_anti")

Members_2025_exist_2023_not_CC_2023 = (
    cc_only_2025.join(exists_2023_not_cc, "cde_id", "inner")
)

# ----------------------------
# 2) Metrics for that cohort in 2023 and 2025
#    Visit definition: distinct (cde_id, VisitDateID, LocationId)
# ----------------------------
tx_2023 = base_2023.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")
tx_2025 = base_2025.join(Members_2025_exist_2023_not_CC_2023, "cde_id", "inner")

m_2023 = tx_2023.agg(
    F.countDistinct("cde_id").alias("Members"),
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2023"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2023"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2023"),
)

m_2025 = tx_2025.agg(
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2025"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2025"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2025"),
)

final = (
    m_2023.crossJoin(m_2025)
    .select(
        "Members",
        "Visits_2023","Visits_2025",
        "Ticket_Revenue_2023","Ticket_Revenue_2025",
        "Concession_Revenue_2023","Concession_Revenue_2025"
    )
)

display(final)

In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ----------------------------
# Compare 2025 cohort vs prior years
# ----------------------------
Y_main = 2025
prior_years = [2018,2019,2020,2021, 2022, 2023, 2024]

start_2025 = Y_main*10000 + 101
end_2025   = (Y_main+1)*10000 + 101

# Base 2025 (only needed cols)
base_2025 = (
    T_trans
    .filter((F.col("VisitDateID") >= start_2025) & (F.col("VisitDateID") < end_2025))
    .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
)

# 2025 cohort: CC-only within 2025 (NULL treated as non-CC, matching your table logic)
non_cc_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") != "Yes")
    .select("cde_id").distinct()
)

cc_only_2025 = (
    base_2025
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id").distinct()
    .join(non_cc_2025, "cde_id", "left_anti")
)

# 2025 metrics for this cohort (same cohort for all comparisons)
tx_2025 = base_2025.join(cc_only_2025, "cde_id", "inner")

m_2025 = tx_2025.agg(
    F.countDistinct("cde_id").alias("Members_2025"),
    F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2025"),
    F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2025"),
    F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2025"),
)

outs = []

for Y_prev in prior_years:
    start_prev = Y_prev*10000 + 101
    end_prev   = (Y_prev+1)*10000 + 101

    base_prev = (
        T_trans
        .filter((F.col("VisitDateID") >= start_prev) & (F.col("VisitDateID") < end_prev))
        .select("cde_id","CineClubFLAG","VisitDateID","LocationId","TicketFLAG","ConcessionFLAG","Net_Revenue")
    )

    # Prior-year status among the 2025 cohort:
    # - exist in prev year (any transaction)
    # - CC in prev year (any CineClubFLAG == Yes)
    exists_prev = base_prev.select("cde_id").distinct()
    cc_any_prev = base_prev.filter(F.col("CineClubFLAG") == "Yes").select("cde_id").distinct()
    exists_prev_not_cc = exists_prev.join(cc_any_prev, "cde_id", "left_anti")

    # Cohort slice: 2025 CC-only who existed in prev year but were NOT CC in prev year
    cohort_slice = cc_only_2025.join(exists_prev_not_cc, "cde_id", "inner")

    # Metrics in prev year for that slice
    tx_prev = base_prev.join(cohort_slice, "cde_id", "inner")

    m_prev = tx_prev.agg(
        F.countDistinct("cde_id").alias("Members_in_slice"),
        F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias(f"Visits_{Y_prev}"),
        F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias(f"Ticket_Revenue_{Y_prev}"),
        F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias(f"Concession_Revenue_{Y_prev}"),
    )

    # Metrics in 2025 for that same slice
    tx_2025_slice = base_2025.join(cohort_slice, "cde_id", "inner")

    m_2025_slice = tx_2025_slice.agg(
        F.countDistinct(F.struct("cde_id","VisitDateID","LocationId")).alias("Visits_2025"),
        F.sum(F.when(F.col("TicketFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Ticket_Revenue_2025"),
        F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.coalesce(F.col("Net_Revenue"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("Concession_Revenue_2025"),
    )

    out = (
        m_prev
        .crossJoin(m_2025_slice)
        .withColumn("Compare_2025_vs", F.lit(Y_prev))
        .select(
            "Compare_2025_vs",
            "Members_in_slice",
            F.col(f"Visits_{Y_prev}").alias("Visits_prior_year"),
            F.col(f"Ticket_Revenue_{Y_prev}").alias("Ticket_Revenue_prior_year"),
            F.col(f"Concession_Revenue_{Y_prev}").alias("Concession_Revenue_prior_year"),
            "Visits_2025",
            "Ticket_Revenue_2025",
            "Concession_Revenue_2025",
        )
    )

    outs.append(out)

df_compare = reduce(lambda a, b: a.unionByName(b), outs)
display(df_compare.orderBy("Compare_2025_vs"))